# Notebook created to explore crossmatch results

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import astropy.units as u
from astropy.wcs import WCS
from matplotlib.patches import Circle

%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
import h5py
from process_fors2.fetchData import readH5FileAttributes

In [ ]:
xmatch_fn = os.path.abspath("./resulting_merge_from_walkthrough.h5")
xmatch_df = readH5FileAttributes(xmatch_fn)
xmatch_df

In [ ]:
f, a = plt.subplots(1, 1)
sns.histplot(data=xmatch_df, x="asep_kids", ax=a, label="KiDS sources")
a.axvline(2, ls="-", c="r", label="Max. KiDS sep.")
a.set_xlabel(r"Separation from FORS2 $[\mathrm{arc\ seconds}]$")
a.legend()

In [ ]:
f, a = plt.subplots(1, 1)
sns.histplot(data=xmatch_df, x="asep_galex", ax=a, label="GALEX sources")
a.axvline(3, ls="-", c="r", label="Max. GALEX sep.")
a.set_xlabel(r"Separation from FORS2 $[\mathrm{arc\ seconds}]$")
a.legend()

In [ ]:
f, a = plt.subplots(1, 1)
sns.histplot(data=xmatch_df, x="redshift", ax=a, label="Spectro-z")
sns.histplot(data=xmatch_df, x="Z_B", ax=a, label="KiDS Z_ML", alpha=0.5)
sns.histplot(data=xmatch_df, x="Z_ML", ax=a, label="KiDS Z_B", alpha=0.3)
a.set_xlabel("Redshift")
a.legend()

In [ ]:
from process_fors2.fetchData import filterCrossMatch, cleanGalexData

In [ ]:
filtered_df = filterCrossMatch(xmatch_fn, 2.0, z_bias=0.1)

In [ ]:
rep, fn = os.path.split(xmatch_fn)
fn, ext = os.path.splitext(fn)
new_fn = f"{fn}_filtered{ext}"
filtered_fn = os.path.join(rep, new_fn)

In [ ]:
clean_galex_df = cleanGalexData(filtered_fn, 3.0)

In [ ]:
clean_galex_df

In [ ]:
clean_galex_df["has_galex"] = clean_galex_df["id_galex"] != "CLEANED"

In [ ]:
f, a = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
sns.histplot(data=clean_galex_df, x="asep_kids", hue="has_galex", ax=a[0])
sns.histplot(data=clean_galex_df, x="redshift", hue="has_galex", ax=a[1])
a[0].set_xlabel(r"Separation from FORS2 $[\mathrm{arc\ seconds}]$")
a[1].set_xlabel(r"Redshift")
f.suptitle("Successful crossmatches with KiDS photometry")

In [ ]:
len(clean_galex_df.index[clean_galex_df["has_galex"]])

In [ ]:
sns.histplot(data=clean_galex_df, x="asep_galex", hue="has_galex")

In [ ]:
f, a = plt.subplots(1, 1)
sns.scatterplot(data=clean_galex_df[clean_galex_df["has_galex"]], x="asep_kids", y="asep_galex")
a.set_xlabel(r"Separation \textit{vs} KiDS $[\mathrm{arcsec}]$")
a.set_ylabel(r"Separation \textit{vs} GALEX $[\mathrm{arcsec}]$")

In [ ]:
len(clean_galex_df)

In [ ]:
CATALOGS = "/home/chevalier/process_fors2/src/data/catalogs/"
images = os.path.join(CATALOGS, "SDSS_images_ugri_005403-282358")
img_to_plot = os.path.join(images, "ADP.2019-02-11T13_02_24.807_TARGET_00_54_03_-28_23_58.fits")

In [ ]:
from astropy.io import fits
from astropy.stats import sigma_clipped_stats

img_hdus = fits.open(img_to_plot)
img_hdr = img_hdus[0].header
img_data = img_hdus[0].data

In [ ]:
img_magsAB = -2.5 * np.log10(img_data)
moy, med, sig = sigma_clipped_stats(img_data)
moyAB, medAB, sigAB = sigma_clipped_stats(img_magsAB)

In [ ]:
img_mask = os.path.join(images, "ADP.2019-02-11T13:02:24.809_TARGET_00:54:03_-28:23:58.fits")
msk_hdus = fits.open(img_to_plot)
msk_hdr = img_hdus[0].header
msk_data = img_hdus[0].data

In [ ]:
%matplotlib widget

In [ ]:
wcs = WCS(img_hdr)
f = plt.figure(figsize=(8, 8), constrained_layout=True)
ax = plt.subplot(projection=wcs)
f.add_axes(ax)
ax.imshow(img_data, vmin=med - 1 * sig, vmax=med + 5 * sig, origin="lower", alpha=0.3)  # , vmin=med-5*sig, vmax=med+5*sig
ax.coords.grid(False, color="white", ls="solid")
ax.coords[0].set_axislabel("Galactic Longitude")
ax.coords[1].set_axislabel("Galactic Latitude")

overlay = ax.get_coords_overlay("fk5")
overlay.grid(True, color="white", ls="dotted")
overlay[0].set_axislabel("Right Ascension (J2000)")
overlay[1].set_axislabel("Declination (J2000)")

ax.scatter(clean_galex_df["ra"], clean_galex_df["dec"], s=16, label="FORS2", transform=ax.get_transform("fk5"), c="orange")
ax.scatter(clean_galex_df["ra_kids"], clean_galex_df["dec_kids"], marker="+", s=49, label="KiDS", transform=ax.get_transform("fk5"), c="r", alpha=0.6)

for ra, dec, asep in zip(clean_galex_df["ra_kids"].values, clean_galex_df["dec_kids"].values, clean_galex_df["asep_kids"].values, strict=True):
    cir = Circle((ra, dec), (asep * u.arcsec).to(u.deg).value, edgecolor="r", facecolor="none", transform=ax.get_transform("fk5"))
    ax.add_patch(cir)
f.legend()  # loc='upper left', bbox_to_anchor=(0.85, -0.02))
plt.show()

In [ ]:
f = plt.figure(figsize=(8, 8), constrained_layout=True)
ax = plt.subplot(projection=wcs)
f.add_axes(ax)
ax.imshow(img_data, vmin=med - 1 * sig, vmax=med + 5 * sig, origin="lower", alpha=0.3)  # , vmin=med-5*sig, vmax=med+5*sig
ax.coords.grid(False, color="white", ls="solid")
ax.coords[0].set_axislabel("Galactic Longitude")
ax.coords[1].set_axislabel("Galactic Latitude")

overlay = ax.get_coords_overlay("fk5")
overlay.grid(True, color="white", ls="dotted")
overlay[0].set_axislabel("Right Ascension (J2000)")
overlay[1].set_axislabel("Declination (J2000)")

ax.scatter(clean_galex_df["ra"], clean_galex_df["dec"], s=16, label="FORS2", transform=ax.get_transform("fk5"), c="orange")
ax.scatter(clean_galex_df["ra_galex"], clean_galex_df["dec_galex"], marker="+", s=49, label="GALEX", transform=ax.get_transform("fk5"), c="r", alpha=0.6)


for ra, dec, asep in zip(clean_galex_df["ra_galex"].values, clean_galex_df["dec_galex"].values, clean_galex_df["asep_galex"].values, strict=True):
    cir = Circle((ra, dec), (asep * u.arcsec).to(u.deg).value, edgecolor="r", facecolor="none", transform=ax.get_transform("fk5"))
    ax.add_patch(cir)
f.legend()  # loc='upper left', bbox_to_anchor=(0.85, -0.02))
plt.show()

In [ ]:
f = plt.figure(figsize=(8, 8), constrained_layout=True)
ax = plt.subplot(projection=wcs)
f.add_axes(ax)
ax.imshow(img_data, vmin=med - 1 * sig, vmax=med + 5 * sig, origin="lower", alpha=0.3)  # , vmin=med-5*sig, vmax=med+5*sig
ax.coords.grid(False, color="white", ls="solid")
ax.coords[0].set_axislabel("Galactic Longitude")
ax.coords[1].set_axislabel("Galactic Latitude")

overlay = ax.get_coords_overlay("fk5")
overlay.grid(True, color="white", ls="dotted")
overlay[0].set_axislabel("Right Ascension (J2000)")
overlay[1].set_axislabel("Declination (J2000)")

ax.scatter(clean_galex_df["ra"], clean_galex_df["dec"], s=16, label="FORS2", transform=ax.get_transform("fk5"), c="orange")

sel = clean_galex_df["has_galex"]
ax.scatter(clean_galex_df[sel]["ra_galex"], clean_galex_df[sel]["dec_galex"], marker="+", s=49, label="GALEX", transform=ax.get_transform("fk5"), c="r", alpha=0.6)

for idx, row in clean_galex_df.iterrows():
    if row["has_galex"]:
        ra, dec, asep = row["ra_galex"], row["dec_galex"], row["asep_galex"]
        cir = Circle((ra, dec), (asep * u.arcsec).to(u.deg).value, edgecolor="r", facecolor="none", transform=ax.get_transform("fk5"))
        ax.add_patch(cir)
f.legend()  # loc='upper left', bbox_to_anchor=(0.85, -0.02))
plt.show()